# Phase 1 — Dense Retriever (MiniLM + FAISS)
This notebook encodes documents & queries using SentenceTransformers (default **all-MiniLM-L6-v2**), builds a FAISS index, retrieves top-k, and evaluates nDCG@10 & MRR@10.

> Matches SOP Step 4 & Step 5 (Dense Retriever + Baseline Evaluation).

In [ ]:
# Optional: install
# !pip install -r requirements.txt

In [15]:
from __future__ import annotations
import json
from pathlib import Path
from typing import List, Tuple, Dict
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss

WORK_DIR = Path("./work")
SUBSET_DIR = WORK_DIR / "subsets" / "beir_trec-covid" / "subset_1"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 100

def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

In [16]:
# 1) Make sure the variable exists and is what you expect
try:
    print("MODEL_NAME =", MODEL_NAME)
except NameError as e:
    raise RuntimeError("MODEL_NAME is not defined in this kernel. "
                       "Re-run the setup cell that defines it.") from e

# 2) Optional: verify the subset path exists and has data
from pathlib import Path
subset_corpus = SUBSET_DIR / "corpus.jsonl"
print("SUBSET_DIR exists:", SUBSET_DIR.exists(), " | corpus.jsonl exists:", subset_corpus.exists())


MODEL_NAME = sentence-transformers/all-MiniLM-L6-v2
SUBSET_DIR exists: True  | corpus.jsonl exists: True


In [3]:
print(repr(MODEL_NAME))  # should print: 'sentence-transformers/all-MiniLM-L6-v2'


'sentence-transformers/all-MiniLM-L6-v2'


In [17]:
from sentence_transformers import SentenceTransformer
import faiss, os
from pathlib import Path

# force writable caches (project-local)
cache_base = Path("./.cache/hf")
for p in [cache_base, cache_base/"hub", cache_base/"transformers", cache_base/"sentence-transformers"]:
    p.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache_base)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(cache_base / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_base / "transformers")
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(cache_base / "sentence-transformers")



In [18]:
# Build FAISS index over corpus
docs = list(read_jsonl(SUBSET_DIR / "corpus.jsonl"))
doc_ids = [r["doc_id"] for r in docs]
doc_texts = [r["text"] for r in docs]

model = SentenceTransformer(MODEL_NAME)
xb = model.encode(
    doc_texts,
    batch_size=256,
    show_progress_bar=False,   # ← turn off widgets progress bar
    convert_to_numpy=True,
    normalize_embeddings=True
)

#xb = model.encode(doc_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
dim = xb.shape[1]

index = faiss.IndexFlatIP(dim)  # cosine with normalized vectors
index.add(xb)
print("FAISS index size:", index.ntotal)

FAISS index size: 55912


In [19]:
# Encode queries and search
queries = list(read_jsonl(SUBSET_DIR / "queries.jsonl"))
qids = [q["qid"] for q in queries]
qtexts = [q["text"] for q in queries]
#xq = model.encode(qtexts, batch_size=256, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
xq = model.encode(
    qtexts,
    batch_size=256,
    show_progress_bar=False,   # ← disable widget bar
    convert_to_numpy=True,
    normalize_embeddings=True
)

D, I = index.search(xq, TOP_K)
run_path = SUBSET_DIR / "run_dense.trec"
with run_path.open("w", encoding="utf-8") as f:
    for qi, qid in enumerate(qids):
        pairs = [(doc_ids[idx], float(D[qi, j])) for j, idx in enumerate(I[qi])]
        pairs.sort(key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(pairs, start=1):
            f.write(f"{qid} Q0 {docid} {rank} {score:.6f} dense\n")
print("Run written:", run_path)

Run written: work/subsets/beir_trec-covid/subset_1/run_dense.trec


In [20]:
# Evaluate nDCG@10, MRR@10
def read_run(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    per_q = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6: 
                continue
            qid, _, docid, _, score, _sys = parts[:6]
            per_q.setdefault(qid, []).append((docid, float(score)))
    for qid in per_q:
        per_q[qid].sort(key=lambda x: x[1], reverse=True)
    return per_q

qrels = {}
for r in read_jsonl(SUBSET_DIR / "qrels.jsonl"):
    qrels.setdefault(str(r["qid"]), {})[str(r["doc_id"])] = int(r["rel"])

def compute_mrr_at_k(run, qrels, k=10):
    mrrs = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        rr = 0.0
        for i, (docid, _) in enumerate(ranked[:k], start=1):
            if relset.get(docid, 0) > 0:
                rr = 1.0 / i
                break
        mrrs.append(rr)
    import numpy as np
    return float(np.mean(mrrs)) if mrrs else 0.0

def compute_ndcg_at_k(run, qrels, k=10):
    import math
    def dcg(rels):
        return sum((rel / math.log2(i + 2)) for i, rel in enumerate(rels))
    ndcgs = []
    for qid, ranked in run.items():
        rels = [1 if qrels.get(qid, {}).get(docid, 0) > 0 else 0 for docid, _ in ranked[:k]]
        idcg = dcg(sorted(rels, reverse=True))
        nd = (dcg(rels) / idcg) if idcg > 0 else 0.0
        ndcgs.append(nd)
    import numpy as np
    return float(np.mean(ndcgs)) if ndcgs else 0.0

run = read_run(run_path)
print({"nDCG@10": compute_ndcg_at_k(run, qrels, 10), "MRR@10": compute_mrr_at_k(run, qrels, 10)})

{'nDCG@10': 0.8755587163755417, 'MRR@10': 0.872549019607843}


In [21]:
# Evaluate nDCG@k and MRR@k with better IDCG and coverage stats
def read_run(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    per_q = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            qid, _, docid, _, score, _sys = parts[:6]
            per_q.setdefault(qid, []).append((docid, float(score)))
    for qid in per_q:
        per_q[qid].sort(key=lambda x: x[1], reverse=True)
    return per_q

# load qrels for the current subset dir
qrels = {}
for r in read_jsonl(SUBSET_DIR / "qrels.jsonl"):
    qrels.setdefault(str(r["qid"]), {})[str(r["doc_id"])] = int(r["rel"])

def mrr_at_k(run, qrels, k=10):
    import numpy as np
    vals = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        rr = 0.0
        for i, (docid, _) in enumerate(ranked[:k], start=1):
            if relset.get(docid, 0) > 0:
                rr = 1.0 / i
                break
        vals.append(rr)
    return float(np.mean(vals)) if vals else 0.0

def ndcg_at_k(run, qrels, k=10, skip_no_rels=True):
    import math, numpy as np
    def dcg(gains):
        return sum(g / math.log2(i+2) for i, g in enumerate(gains))
    vals = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        R = sum(1 for v in relset.values() if v > 0)  # total relevant in subset
        if R == 0:
            if skip_no_rels:
                continue
            else:
                vals.append(0.0); continue
        gains = [1 if relset.get(docid, 0) > 0 else 0 for docid, _ in ranked[:k]]
        ideal = [1]*min(k, R) + [0]*max(0, k - min(k, R))
        idcg = dcg(ideal)
        vals.append((dcg(gains)/idcg) if idcg > 0 else 0.0)
    return float(np.mean(vals)) if vals else 0.0

# helpful diagnostics
def evaluation_report(run, qrels, k=10):
    total_q = len(run)
    with_rels = sum(1 for q in run if any(v>0 for v in qrels.get(q, {}).values()))
    at_least_one_hit = sum(
        1 for qid, ranked in run.items()
        if any(qrels.get(qid, {}).get(docid, 0)>0 for docid, _ in ranked[:k])
    )
    return {
        "queries_in_run": total_q,
        "queries_with_rels_in_subset": with_rels,
        "queries_with_at_least_one_hit@k": at_least_one_hit
    }

K = 10
run = read_run(run_path)
print(evaluation_report(run, qrels, k=K))
print({
    "nDCG@10": round(ndcg_at_k(run, qrels, k=K, skip_no_rels=True), 4),
    "MRR@10":  round(mrr_at_k(run, qrels, k=K), 4)
})


{'queries_in_run': 17, 'queries_with_rels_in_subset': 17, 'queries_with_at_least_one_hit@k': 16}
{'nDCG@10': 0.7491, 'MRR@10': 0.8725}
